1: Imports and Data Loading

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Load the CSV file
# Ensure 'credit_risk_data.csv' is in the same directory as your notebook
df = pd.read_csv('credit_risk_dataset.csv')

# Quick look at the data
print("Data Shape:", df.shape)
df.head()

Data Shape: (32581, 12)


,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4


2: Preprocessing and Cleaning

In [2]:
# 1️⃣ Remove missing values FIRST
df = df.dropna()

# 2️⃣ Encode categorical columns
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

cat_cols = ['person_home_ownership',
            'loan_intent',
            'loan_grade',
            'cb_person_default_on_file']

for col in cat_cols:
    df[col] = le.fit_transform(df[col])

# 3️⃣ Split features and target
X = df.drop('loan_status', axis=1)
y = df['loan_status']

# 4️⃣ Train-test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 5️⃣ Scale numerical features
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Ready for training ✅")

Ready for training ✅


3: ANN Architecture (Functional API)

In [3]:
# Define the Input Layer
# shape=(11,) because we have 11 feature columns
# inputs = Input(shape=(X_train_scaled.shape[1],), name='Input_Layer')
inputs = Input(shape=(11,), name='Input_Layer')

# Hidden Layer 1
x = Dense(64, activation='relu', name='Dense_1')(inputs)
x = Dropout(0.2)(x) # Helps prevent overfitting

# Hidden Layer 2
x = Dense(32, activation='relu', name='Dense_2')(x)

# Hidden Layer 3
x = Dense(16, activation='relu', name='Dense_3')(x)

# Output Layer
# Sigmoid is used for binary classification (Probability of 0 or 1)
outputs = Dense(1, activation='sigmoid', name='Output_Layer')(x)

# Create the Model object
model = Model(inputs=inputs, outputs=outputs, name='Loan_Risk_Model')

# Compile
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.summary()
print(type(x))

Model: "Loan_Risk_Model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Input_Layer (InputLayer)        │ (None, 11)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Dense_1 (Dense)                 │ (None, 64)             │           768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Dense_3 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Output_Layer (Dense)            │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,393 (13.25 KB)

 Trainable params: 3,393 (13.25 KB)

 Non-trainable params: 0 (0.00 B)

<class 'keras.src.backend.common.keras_tensor.KerasTensor'>


4: Training and Evaluation

In [4]:
# Train the model
# We use a small batch_size because your sample data is small
history = model.fit(
    X_train_scaled,
    y_train,
    epochs=30,
    batch_size=4,
    validation_split=0.2,
    verbose=1
)

# Evaluate on the Test Set
loss, accuracy = model.evaluate(X_test_scaled, y_test)
print(f"\nTest Accuracy: {accuracy * 100:.2f}%")

Epoch 1/30
4582/4582 ━━━━━━━━━━━━━━━━━━━━ 29s 6ms/step - accuracy: 0.8282 - loss: 0.3980 - val_accuracy: 0.8712 - val_loss: 0.3230
Epoch 2/30
4582/4582 ━━━━━━━━━━━━━━━━━━━━ 25s 2ms/step - accuracy: 0.8653 - loss: 0.3297 - val_accuracy: 0.8734 - val_loss: 0.3113
Epoch 3/30
4582/4582 ━━━━━━━━━━━━━━━━━━━━ 12s 3ms/step - accuracy: 0.8765 - loss: 0.3145 - val_accuracy: 0.8861 - val_loss: 0.2918
Epoch 4/30
4582/4582 ━━━━━━━━━━━━━━━━━━━━ 12s 3ms/step - accuracy: 0.8832 - loss: 0.2989 - val_accuracy: 0.8900 - val_loss: 0.2841
Epoch 5/30
4582/4582 ━━━━━━━━━━━━━━━━━━━━ 12s 3ms/step - accuracy: 0.8890 - loss: 0.2925 - val_accuracy: 0.8931 - val_loss: 0.2797
Epoch 6/30
4582/4582 ━━━━━━━━━━━━━━━━━━━━ 11s 2ms/step - accuracy: 0.8906 - loss: 0.2840 - val_accuracy: 0.8904 - val_loss: 0.2786
Epoch 7/30
4582/4582 ━━━━━━━━━━━━━━━━━━━━ 13s 3ms/step - accuracy: 0.8887 - loss: 0.2835 - val_accuracy: 0.8976 - val_loss: 0.2759
Epoch 8/30
4582/4582 ━━━━━━━━━━━━━━━━━━━━ 12s 3ms/step - accuracy: 0.8924 - loss: 0